In [1]:
# ==========================================
#   IMPORT LIBRARIES & SETUP
# ==========================================
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'   # implicit recommends this for ALS

%load_ext autoreload
%autoreload 2

from metrics import evaluate
import pandas as pd 
import numpy as np
from scipy.sparse import csr_matrix
import implicit
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import CosineRecommender

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## DATA LOADING

In [2]:
articles=pd.read_parquet('data/articles_processed.parquet')
customers=pd.read_parquet('data/customers_processed.parquet')
transactions=pd.read_parquet('data/transactions_processed.parquet')


In [3]:
#=================================================================================
#                        MAPPING
#=================================================================================

unique_customers = customers['customer_id'].unique()
unique_articles = articles['article_id'].unique()

# Mappings
customer2idx = {c: i for i, c in enumerate(unique_customers)}
article2idx = {a: i for i, a in enumerate(unique_articles)}

# Reverse Mappings 
idx2customer = {i: c for c, i in customer2idx.items()}
idx2article = {i: a for a, i in article2idx.items()}

# Mapping transactions-ի վրա
transactions['user_idx'] = transactions['customer_id'].map(customer2idx)
transactions['item_idx'] = transactions['article_id'].map(article2idx)

# Check: no unmapped IDs
print(transactions['user_idx'].isna().sum(), transactions['item_idx'].isna().sum())

0 0


In [4]:
#================================================================================
#                  TRAIN-VALIDATION-TEST SPLIT
#================================================================================
last_date = transactions['t_dat'].max()
test_start = last_date - pd.Timedelta(days=6)    # last week
val_start = test_start - pd.Timedelta(days=7)    # the week before last

train = transactions[transactions['t_dat'] < val_start]
val = transactions[(transactions['t_dat'] >= val_start) & (transactions['t_dat'] < test_start)]
test = transactions[transactions['t_dat'] >= test_start]

## Collaborative Filtering

In [5]:
#=================================================================================
#                              CSR Matrix
#=================================================================================

train_user_item = train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')

rows = train_user_item['user_idx'].values
cols = train_user_item['item_idx'].values
weights = train_user_item['weight'].values.astype(np.float32)

n_users = len(customer2idx)
n_items = len(article2idx)


user_item_matrix = csr_matrix((weights, (rows, cols)), shape=(n_users, n_items))


In [6]:

recent_popular = train[train['t_dat'] >= train['t_dat'].max() - pd.Timedelta(days=6)]
fallback_top12 = recent_popular.groupby('article_id').size().nlargest(12).index.tolist()

def predict_implicit_model(model, val_df, k=12):
    
    actuals_dict = val_df.groupby('customer_id')['article_id'].unique().apply(set).to_dict()
    val_customers = list(actuals_dict.keys())

    user_idxs = np.array([customer2idx[c_id] for c_id in val_customers])

    has_history = np.asarray(user_item_matrix[user_idxs].getnnz(axis=1)) > 0

    ids, scores = model.recommend(
        user_idxs,
        user_item_matrix[user_idxs],
        N=k,
        filter_already_liked_items=False
    )

    predictions = {}
    for i, c_id in enumerate(val_customers):
        if has_history[i]:
            recs = [idx2article[int(idx)] for idx in ids[i] if idx >= 0]
        else:
            recs = []  # cold customer
        fill = [p for p in fallback_top12 if p not in recs]
        predictions[c_id] = (recs + fill)[:k]

    return predictions, actuals_dict

In [7]:
#=================================================================================
#                   Item-Item Cosine Similarity
#=================================================================================

cosine_model = CosineRecommender(K=20)
cosine_model.fit(user_item_matrix)
print("Cosine Similarity model successfully fitted!")
print(cosine_model.similarity.shape)

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.15831613540649414 seconds
  warnings.warn(
100%|██████████| 105542/105542 [00:06<00:00, 17385.40it/s]


Cosine Similarity model successfully fitted!
(105542, 105542)


In [8]:
cosine_preds, actuals_dict = predict_implicit_model(cosine_model, val, k=12)

total_catalog_items = articles['article_id'].nunique()
sales_series = train['article_id'].value_counts()

total_sales = len(train)

novelty_dict = (
    (-np.log2(sales_series / total_sales)).to_dict()
)

cosine_results = evaluate(cosine_preds, actuals_dict, total_catalog_items,novelty_dict, k=12)

print("--- Cosine Similarity Results ---")
for metric_name, value in cosine_results.items():
    print(f"{metric_name}: {value:.5f}")

--- Cosine Similarity Results ---
Precision@12: 0.00497
Recall@12: 0.02654
MAP@12: 0.01075
NDCG@12: 0.01697
Novelty@12: 14.71539
Coverage: 0.69442


In [9]:
val_results = pd.read_csv('results/val_results.csv', index_col=0)

cosine_row = pd.DataFrame(cosine_results, index=['Cosine kNN (K=20)'])
val_results = pd.concat([val_results, cosine_row])

val_results.to_csv('results/val_results.csv')
val_results

,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage
Global Popularity,0.002341,0.008971,0.003444,0.006318,10.216682,0.000114
Recent Popularity (7d),0.004933,0.020905,0.006604,0.012605,13.269735,0.000114
Repeat + Popularity Fallback,0.004374,0.023520,0.009352,0.015019,14.318081,0.569129
Top 5 Repeat + Popularity,0.005515,0.026711,0.009696,0.016547,13.521146,0.427062
Cosine kNN (K=20),0.004973,0.026540,0.010754,0.016974,14.715392,0.694415


* Baseline-ը ցույց է տվել , որ հաճախորդի գնումները հաշվի առնելը լավ արդյունք է տալիս։Ստուգենք val set-ում գնված ապրանքները train-ի վերջին 4 շաբաթվա գնումներից են թե ոչ

In [12]:
val_pairs = val[['user_idx', 'item_idx']].drop_duplicates()
all_hist = train[['user_idx', 'item_idx']].drop_duplicates()

cutoff = train['t_dat'].max() - pd.Timedelta(days=27)   # last 4 weeks of train
recent_hist = train.loc[train['t_dat'] >= cutoff, ['user_idx', 'item_idx']].drop_duplicates()

share_all = len(val_pairs.merge(all_hist)) / len(val_pairs)
share_recent = len(val_pairs.merge(recent_hist)) / len(val_pairs)

print(len(val_pairs), round(share_all, 4), round(share_recent, 4))

227910 0.0393 0.0228


### Repeat գնումների վերլուծություն և hybrid մոդել
 Validation շաբաթվա (հաճախորդ, ապրանք) զույգերի միայն 3.93%-ն է հաճախորդը նախկինում գնել train-ի ողջ պատմության մեջ, այսինքն՝ գնումների մոտ 96%-ը նոր ապրանք է։ Եթե հաշվենք միայն train-ի վերջին 4 շաբաթը, կազմում է 2.28% , այսինքն՝ repeat գնումների մոտ 60%-ը գալիս է վերջին 4 շաբաթվա պատմությունից։

Կառուցենք մոդել ,որ հաճախորդի վերջին 4 շաբաթվա ամենից շատ գնած 2 կամ 3 ապրանքը դնում է ցանկի սկզբում, իսկ մնացած տեղերը լրացվում են Cosine-ով


In [13]:
# --- last 4 weeks of train: how many times each customer bought each article ---
cutoff = train['t_dat'].max() - pd.Timedelta(days=27)   # last 4 weeks of train
recent_train = train[train['t_dat'] >= cutoff]

recent_ui = recent_train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')
recent_ui = recent_ui.sort_values(['user_idx', 'weight'], ascending=[True, False])

cosine_preds, actuals_dict = predict_implicit_model(cosine_model, val, k=12)

model_results = {}

for n_repeat in [2, 3, 4, 5, 6, 8, 12]:
    top_repeat = recent_ui.groupby('user_idx').head(n_repeat)
    repeat_by_user = top_repeat.groupby('user_idx')['item_idx'].apply(list).to_dict()

    hybrid_preds = {}
    for c_id, model_recs in cosine_preds.items():
        repeat = [idx2article[int(i)] for i in repeat_by_user.get(customer2idx[c_id], [])]
        fill = [r for r in model_recs if r not in repeat]   # no duplicates
        hybrid_preds[c_id] = (repeat + fill)[:12]

    model_results[f'Hybrid: {n_repeat} recent repeat + Cosine'] = evaluate(
        hybrid_preds, actuals_dict, total_catalog_items, novelty_dict, k=12)
    print(n_repeat, 'done')



val_results = pd.concat([val_results, pd.DataFrame(model_results).T])
val_results.to_csv('results/val_results.csv')
val_results.round(5)

2 done
3 done
4 done
5 done
6 done
8 done
12 done


,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage
Global Popularity,0.00234,0.00897,0.00344,0.00632,10.21668,0.00011
Recent Popularity (7d),0.00493,0.02091,0.00660,0.01260,13.26973,0.00011
Repeat + Popularity Fallback,0.00437,0.02352,0.00935,0.01502,14.31808,0.56913
Top 5 Repeat + Popularity,0.00551,0.02671,0.00970,0.01655,13.52115,0.42706
Cosine kNN (K=20),0.00497,0.02654,0.01075,0.01697,14.71539,0.69442
Hybrid: 2 recent repeat + Cosine,0.00603,0.03300,0.01887,0.02583,14.74163,0.69046
Hybrid: 3 recent repeat + Cosine,0.00643,0.03523,0.01986,0.02733,14.75861,0.68822
Hybrid: 4 recent repeat + Cosine,0.00679,0.03725,0.02049,0.02841,14.77520,0.68618
Hybrid: 5 recent repeat + Cosine,0.00707,0.03855,0.02083,0.02908,14.78923,0.68380
Hybrid: 6 recent repeat + Cosine,0.00732,0.03969,0.02109,0.02960,14.80158,0.68175
